# Métricas V3 — Zonas (88 zonas, radio 75 mi)

Calcula cobertura y deadhead para **V3 (zonas)** con la misma métrica usada en `3_comparacion_hubs.ipynb` para M1 y M2, para completar el criterio 1.2 del convenio.

### Metadata del análisis
| Campo | Valor |
|---|---|
| N registros | 57,573,489 |
| Ventana de fechas | Feb–Ago 2025 |
| Tipo de validación | Within-sample (misma muestra que M1/M2) |
| Radio V3 | 75 millas |
| Zonas | 88 |

### Inputs esperados
| Variable | Ruta |
|---|---|
| Asignación M1 (pre-computada) | `loads_assign_m1.parquet` |
| Centroides V3 (zonas) | `ZONES_EXCEL` → sheet `zones id` |
| Hubs → Zona mapping | `ZONES_EXCEL` → sheet `zones hub` |

### Outputs
- `coords_assign_v3.parquet` — lookup table: coordenada → (near_zone, dist_mi, zones_75m)
- `loads_assign_v3.parquet` — cargas + asignación V3 origen + destino
- `tabla_comparativa_v1v2v3.csv` — tabla resumen para criterio 1.2

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from pathlib import Path
import openpyxl
import gc

# ── Paths ────────────────────────────────────────────────────────────────────
ZONES_EXCEL    = Path('../utils/hubs_id.xlsx')
LOADS_M1_PATH  = 'loads_assign_m1.parquet'   # generado en 3_comparacion_hubs.ipynb
OUT_COORDS_V3  = 'coords_assign_v3.parquet'
OUT_LOADS_V3   = 'loads_assign_v3.parquet'
OUT_TABLE      = 'tabla_comparativa_v1v2v3.csv'

# ── Parámetros ────────────────────────────────────────────────────────────────
RADIO_V3_MI = 75.0           # radio de cobertura V3 en millas
EARTH_R_MI  = 3958.8         # radio terrestre en millas
MI_TO_RAD   = 1.0 / EARTH_R_MI  # factor de conversión para BallTree (haversine)

print('Librerías cargadas.')

## 1. Centroides de zonas V3 desde Excel

In [ ]:
# Sheet 1 (idx 0) = 'hubs_id.csv'   → 194 hubs V1
# Sheet 2 (idx 1) = 'zones id'      → 88 centroides V3
# Sheet 3 (idx 2) = 'zones hub'     → mapping hub → zona
print('Sheets disponibles:', pd.ExcelFile(ZONES_EXCEL).sheet_names)

zones        = pd.read_excel(ZONES_EXCEL, sheet_name=1, engine='openpyxl')
hubs_to_zone = pd.read_excel(ZONES_EXCEL, sheet_name=2, engine='openpyxl')

print('Columnas zones id:  ', zones.columns.tolist())
print('Columnas zones hub: ', hubs_to_zone.columns.tolist())

# zones hub: columnas 'id' (hub_id) y 'zone' (numero de zona 1–88)
hubs_to_zone = (
    hubs_to_zone[['id', 'zone']]
    .rename(columns={'id': 'hub_id', 'zone': 'zone_num'})
    .assign(hub_id=lambda d: d['hub_id'].astype(int),
            zone_num=lambda d: d['zone_num'].astype(int))
)

print(f'\nZonas V3: {len(zones)}')
print(f'Hubs con zona asignada: {len(hubs_to_zone)}')
print(f'Zonas únicas en hubs: {hubs_to_zone["zone_num"].nunique()}')
print(zones[['zone_id', 'Zone_name', 'lat', 'lng', 'dhd_q1', 'dhd_q2', 'dhd_q3']].head())

## 2. BallTree sobre centroides de zonas

Para cada coordenada del dataset, encontramos la zona más cercana y calculamos la distancia haversine en millas.

In [ ]:
# Preparar centroides en radianes para BallTree haversine
zone_lats = zones['lat'].astype(float).to_numpy()
zone_lngs = zones['lng'].astype(float).to_numpy()
zone_ids  = np.array(zones['zone_id'].tolist(), dtype=object)   # numpy object array, no ArrowStringArray

zone_coords_rad = np.column_stack([
    np.deg2rad(zone_lats),
    np.deg2rad(zone_lngs)
])

tree = BallTree(zone_coords_rad, metric='haversine')
print(f'BallTree construido con {len(zone_coords_rad)} centroides de zona.')
print(f'Ejemplo zone_ids: {zone_ids[:3]}')

## 3. Coordenadas únicas del dataset de cargas

Extraemos las coordenadas únicas (origen y destino) de `loads_assign_m1.parquet` para construir un lookup table, siguiendo el mismo patrón de M1/M2.

In [ ]:
# Extraer coordenadas únicas de origen y destino
print('Cargando coordenadas únicas...')

orig_coords = (
    pl.scan_parquet(LOADS_M1_PATH)
    .select(['LatOrigin', 'LngOrigin'])
    .filter(pl.col('LatOrigin').is_not_null() & pl.col('LngOrigin').is_not_null())
    .unique()
    .collect()
)
dest_coords = (
    pl.scan_parquet(LOADS_M1_PATH)
    .select(['LatDestination', 'LngDestination'])
    .filter(pl.col('LatDestination').is_not_null() & pl.col('LngDestination').is_not_null())
    .unique()
    .collect()
    .rename({'LatDestination': 'LatOrigin', 'LngDestination': 'LngOrigin'})  # unificar nombre
)

all_coords = (
    pl.concat([orig_coords, dest_coords])
    .unique()
    .sort(['LatOrigin', 'LngOrigin'])
)

print(f'Coordenadas únicas (origen): {len(orig_coords):,}')
print(f'Coordenadas únicas (destino): {len(dest_coords):,}')
print(f'Coordenadas únicas total: {len(all_coords):,}')
del orig_coords, dest_coords; gc.collect()

## 4. Asignación de coordenadas a zonas (BallTree query)

Para cada coordenada, buscamos la zona más cercana y todas las zonas dentro de 75 millas.

In [ ]:
RADIO_RAD = RADIO_V3_MI * MI_TO_RAD   # 75 mi en radianes

lats = all_coords['LatOrigin'].to_numpy()
lngs = all_coords['LngOrigin'].to_numpy()
query_rad = np.column_stack([np.deg2rad(lats), np.deg2rad(lngs)])

# 1. Zona más cercana + distancia
print('Consultando BallTree (nearest)...')
dists_rad, idxs = tree.query(query_rad, k=1, return_distance=True)
dists_mi  = dists_rad.ravel() * EARTH_R_MI
near_idx  = idxs.ravel()
near_zone = zone_ids[near_idx]           # e.g. 'Z_3'

# 2. Todas las zonas dentro de 75 mi
print('Consultando BallTree (radius)...')
ind_within = tree.query_radius(query_rad, r=RADIO_RAD)
zones_75m  = [str([zone_ids[i] for i in row]) for row in ind_within]  # lista como string

coords_v3 = (
    all_coords
    .with_columns([
        pl.Series('near_zone_v3', near_zone),
        pl.Series('dist_mi_v3',   dists_mi.round(4)),
        pl.Series('zones_75m_v3', zones_75m),
    ])
    .rename({'LatOrigin': 'lat', 'LngOrigin': 'lng'})
)

coords_v3.write_parquet(OUT_COORDS_V3)
print(f'Guardado: {OUT_COORDS_V3}  ({len(coords_v3):,} filas)')

cubiertos = (coords_v3['dist_mi_v3'] <= RADIO_V3_MI).sum()
print(f'Coordenadas cubiertas (≤75 mi): {cubiertos:,} / {len(coords_v3):,} = {100*cubiertos/len(coords_v3):.1f}%')
coords_v3.head()

## 5. Parquet de cargas con asignación V3

In [ ]:
# Preparar lookup tables para origen y destino
coords_pl = pl.read_parquet(OUT_COORDS_V3)

v3_orig = (
    coords_pl
    .rename({
        'lat':          'lat_orig',
        'lng':          'lng_orig',
        'near_zone_v3': 'near_zone_v3_orig',
        'dist_mi_v3':   'dist_mi_v3_orig',
        'zones_75m_v3': 'zones_75m_v3_orig',
    })
)
v3_dest = (
    coords_pl
    .rename({
        'lat':          'lat_dest',
        'lng':          'lng_dest',
        'near_zone_v3': 'near_zone_v3_dest',
        'dist_mi_v3':   'dist_mi_v3_dest',
        'zones_75m_v3': 'zones_75m_v3_dest',
    })
)

loads_v3 = (
    pl.scan_parquet(LOADS_M1_PATH)
    .with_columns([
        pl.col('LatOrigin').cast(pl.Float64),
        pl.col('LngOrigin').cast(pl.Float64),
        pl.col('LatDestination').cast(pl.Float64),
        pl.col('LngDestination').cast(pl.Float64),
    ])
    .join(
        v3_orig.lazy().with_columns([
            pl.col('lat_orig').cast(pl.Float64),
            pl.col('lng_orig').cast(pl.Float64),
        ]),
        left_on=['LatOrigin', 'LngOrigin'],
        right_on=['lat_orig', 'lng_orig'],
        how='left'
    )
    .join(
        v3_dest.lazy().with_columns([
            pl.col('lat_dest').cast(pl.Float64),
            pl.col('lng_dest').cast(pl.Float64),
        ]),
        left_on=['LatDestination', 'LngDestination'],
        right_on=['lat_dest', 'lng_dest'],
        how='left'
    )
    .collect()
)

loads_v3.write_parquet(OUT_LOADS_V3)
print(f'Guardado: {OUT_LOADS_V3}  ({len(loads_v3):,} filas)')

## 6. Cobertura V3 a 75 millas

Métrica: porcentaje de cargas cuya coordenada de origen (outbound) o destino (inbound) está dentro de 75 mi de algún centroide de zona.

In [ ]:
total = len(loads_v3)

# Cobertura: dist_mi ≤ 75 (carga tiene zona dentro de 75 mi)
cov_orig = (loads_v3['dist_mi_v3_orig'] <= RADIO_V3_MI).sum()
cov_dest = (loads_v3['dist_mi_v3_dest'] <= RADIO_V3_MI).sum()
cov_both = (
    (loads_v3['dist_mi_v3_orig'] <= RADIO_V3_MI) &
    (loads_v3['dist_mi_v3_dest'] <= RADIO_V3_MI)
).sum()
cov_any  = (
    (loads_v3['dist_mi_v3_orig'] <= RADIO_V3_MI) |
    (loads_v3['dist_mi_v3_dest'] <= RADIO_V3_MI)
).sum()

con_orig = loads_v3['LatOrigin'].is_not_null().sum()
con_dest = loads_v3['LatDestination'].is_not_null().sum()

print(f'N total de cargas:              {total:>14,}')
print(f'Con coordenada origen:          {con_orig:>14,}  ({100*con_orig/total:.1f}%)')
print(f'Con coordenada destino:         {con_dest:>14,}  ({100*con_dest/total:.1f}%)')
print()
print(f'Cobertura origen (outbound):    {cov_orig:>14,}  ({100*cov_orig/total:.1f}%  de total)')
print(f'Cobertura destino (inbound):    {cov_dest:>14,}  ({100*cov_dest/total:.1f}%  de total)')
print(f'Cobertura ambos extremos:       {cov_both:>14,}  ({100*cov_both/total:.1f}%  de total)')
print(f'Cobertura al menos un extremo:  {cov_any:>14,}  ({100*cov_any/total:.1f}%  de total)')

## 7. Deadhead V3 — distancia al centroide de zona más cercano

Para las cargas cubiertas, calculamos la distribución de distancias al centroide de zona.

In [ ]:
# Deadhead: dist al centroide de zona más cercano para cargas cubiertas (origen)
dhd_orig_covered = (
    loads_v3
    .filter(pl.col('dist_mi_v3_orig') <= RADIO_V3_MI)
    ['dist_mi_v3_orig']
)
dhd_dest_covered = (
    loads_v3
    .filter(pl.col('dist_mi_v3_dest') <= RADIO_V3_MI)
    ['dist_mi_v3_dest']
)

# Combinado (todos los puntos cubiertos, origen y destino)
dhd_all = pl.concat([dhd_orig_covered, dhd_dest_covered])

print('Deadhead V3 (dist a centroide de zona, cargas cubiertas ≤75mi):')
print(f'  Origen  — Q1={dhd_orig_covered.quantile(0.25):.1f} mi  Q2={dhd_orig_covered.quantile(0.5):.1f} mi  Q3={dhd_orig_covered.quantile(0.75):.1f} mi  Media={dhd_orig_covered.mean():.1f} mi')
print(f'  Destino — Q1={dhd_dest_covered.quantile(0.25):.1f} mi  Q2={dhd_dest_covered.quantile(0.5):.1f} mi  Q3={dhd_dest_covered.quantile(0.75):.1f} mi  Media={dhd_dest_covered.mean():.1f} mi')
print(f'  Global  — Q1={dhd_all.quantile(0.25):.1f} mi  Q2={dhd_all.quantile(0.5):.1f} mi  Q3={dhd_all.quantile(0.75):.1f} mi  Media={dhd_all.mean():.1f} mi')

# Distribución por tramo
print('\nDistribución de deadhead V3 (% de cargas cubiertas por tramo):')
total_cov = len(dhd_all)
for lo, hi in [(0, 25), (25, 50), (50, 75)]:
    n = ((dhd_all >= lo) & (dhd_all < hi)).sum()
    print(f'  {lo:>3}–{hi:<3} mi: {n:>12,}  ({100*n/total_cov:.1f}%)')

## 8. Tabla comparativa V1 / V2 / V3

Tabla unificada con misma métrica de cobertura (% cargas cubiertas) sobre el mismo dataset (Feb–Ago 2025, N=57,573,489), para criterio 1.2 del convenio.

In [ ]:
# ── Métricas M1 y M2 de 3_comparacion_hubs.ipynb ────────────────────────────
# (ya calculadas; los valores vienen del output de ese notebook)
M1_COV_ORIG = 83.7   # % de total
M1_COV_DEST = 83.4
M1_DHD_Q2   = 32.3   # mediana deadhead (mi)

M2_COV_ORIG = 81.2
M2_COV_DEST = 83.0
M2_DHD_Q2   = 24.7

# ── Métricas V3 calculadas arriba ────────────────────────────────────────────
v3_cov_orig = round(100 * cov_orig / total, 1)
v3_cov_dest = round(100 * cov_dest / total, 1)
v3_dhd_q2   = round(float(dhd_all.quantile(0.5)), 1)

# Nota V2: 192 hubs totales = 188 KDE+DBSCAN orgánicos + 4 imputados (NJ, CT, RI, DE)
# Los 4 imputados se agregaron para garantizar cobertura en el pipeline de indicadores.
tabla = pd.DataFrame([
    {
        'Versión':           'V1 — K-Medoids',
        'Nodos':             194,
        'Radio (mi)':        50,
        '% origen cubiert':  M1_COV_ORIG,
        '% destino cubiert': M1_COV_DEST,
        'Deadhead Q2 (mi)':  M1_DHD_Q2,
        'Dataset':           'Feb–Ago 2025',
        'N':                 57_573_489,
    },
    {
        'Versión':           'V2 — KDE+DBSCAN',
        'Nodos':             192,   # 188 orgánicos + 4 imputados (NJ, CT, RI, DE)
        'Radio (mi)':        50,
        '% origen cubiert':  M2_COV_ORIG,
        '% destino cubiert': M2_COV_DEST,
        'Deadhead Q2 (mi)':  M2_DHD_Q2,
        'Dataset':           'Feb–Ago 2025',
        'N':                 57_573_489,
    },
    {
        'Versión':           'V3 — Zonas',
        'Nodos':             88,
        'Radio (mi)':        75,
        '% origen cubiert':  v3_cov_orig,
        '% destino cubiert': v3_cov_dest,
        'Deadhead Q2 (mi)':  v3_dhd_q2,
        'Dataset':           'Feb–Ago 2025',
        'N':                 57_573_489,
    },
])

tabla.to_csv(OUT_TABLE, index=False)
print('Tabla comparativa V1/V2/V3:')
print(tabla.to_string(index=False))
print(f'\nGuardada en: {OUT_TABLE}')

## 9. Nota metodológica — comparabilidad

> **Nota**: La comparación V1/V2 usa radio de **50 mi**; V3 usa **75 mi** (su radio operativo natural). Para leer la tabla correctamente: V3 cubre más cargas en parte porque su radio es mayor. Una comparación con radio fijo de 50 mi para V3 se puede obtener reemplazando `RADIO_V3_MI = 50.0` al principio del notebook y volviendo a correr la celda 6.

In [ ]:
# Cobertura V3 a 50 mi (para comparación apple-to-apple)
cov_orig_50 = (loads_v3['dist_mi_v3_orig'] <= 50.0).sum()
cov_dest_50 = (loads_v3['dist_mi_v3_dest'] <= 50.0).sum()

print('Cobertura V3 a radio fijo de 50 mi (apple-to-apple con V1/V2):')
print(f'  Origen:  {100*cov_orig_50/total:.1f}%  ({cov_orig_50:,} cargas)')
print(f'  Destino: {100*cov_dest_50/total:.1f}%  ({cov_dest_50:,} cargas)')
print()
print('Comparación con radio fijo 50 mi:')
print(f'  V1 origen: {M1_COV_ORIG}%  |  V2 origen: {M2_COV_ORIG}%  |  V3 origen (50mi): {100*cov_orig_50/total:.1f}%')
print(f'  V1 destino: {M1_COV_DEST}%  |  V2 destino: {M2_COV_DEST}%  |  V3 destino (50mi): {100*cov_dest_50/total:.1f}%')

## 10. Mapa Folium — V1 / V2 / V3 (capas independientes)

Mapa con tres capas togglables: hubs V1 (K-Medoids, azul), hubs V2 (KDE+DBSCAN, naranja) y centroides de zonas V3 (rojo). Se exporta como HTML para no ocupar espacio en el notebook.

In [ ]:
import folium
import os

m = folium.Map(location=[38, -96], zoom_start=4, tiles='OpenStreetMap')

# ── Capa V1 (K-Medoids) ───────────────────────────────────────────────────────
fg_v1 = folium.FeatureGroup(name='V1 — K-Medoids (194 hubs)', show=True)
hubs_v1 = pd.read_excel(ZONES_EXCEL, sheet_name=0, engine='openpyxl')
for _, row in hubs_v1.iterrows():
    name = row.get('new_name') or row.get('old_name') or str(row['id'])
    folium.CircleMarker(
        [row['lat'], row['lng']],
        radius=5, color='steelblue', fill=True, fill_opacity=0.7,
        tooltip=f"V1 ({row.get('state', '')}): {name}"
    ).add_to(fg_v1)

# ── Capa V2 (KDE+DBSCAN) ─────────────────────────────────────────────────────
fg_v2 = folium.FeatureGroup(name='V2 — KDE+DBSCAN', show=True)
m2_path = 'final_results_optimized.csv' if os.path.exists('final_results_optimized.csv') else 'final_results.csv'
hubs_v2 = pd.read_csv(m2_path).dropna(subset=['lat', 'lng'])
for _, row in hubs_v2.iterrows():
    name = row.get('new_name') or str(row.get('hub_id', ''))
    folium.CircleMarker(
        [row['lat'], row['lng']],
        radius=5, color='darkorange', fill=True, fill_opacity=0.7,
        tooltip=f"V2 ({row.get('state', '')}): {name}"
    ).add_to(fg_v2)

# ── Capa V3 (Zonas) ──────────────────────────────────────────────────────────
fg_v3 = folium.FeatureGroup(name='V3 — Zonas (88 zonas)', show=True)
for _, row in zones.iterrows():
    folium.CircleMarker(
        [row['lat'], row['lng']],
        radius=7, color='crimson', fill=True, fill_opacity=0.8,
        tooltip=f"V3 Zona {row['zone_id']}: {row.get('Zone_name', '')}"
    ).add_to(fg_v3)

fg_v1.add_to(m)
fg_v2.add_to(m)
fg_v3.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

OUT_MAP = 'mapa_v1_v2_v3.html'
m.save(OUT_MAP)
print(f'Mapa exportado: {OUT_MAP}')